In [33]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import sys
import numpy as np
import os
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import mlflow
import mlflow.pytorch
import random
import glob
sys.path.append("../../MainProject/Assignment9")
from assignment9_functions import *


# Set device
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Using device: mps


In [34]:
def split_by_video(train_files_kinect, train_files_mp, test_files_kinect, test_files_mp, 
                   val_ratio=0.15, random_seed=42):
    """
    Split training videos into train/val while keeping test separate
    """
    import random
    
    # Pair the training files
    train_pairs = list(zip(train_files_kinect, train_files_mp))
    
    # Shuffle
    random.seed(random_seed)
    random.shuffle(train_pairs)
    
    # Split by VIDEO (not by frame!)
    n_train_videos = len(train_pairs)
    n_val_videos = int(n_train_videos * val_ratio)
    
    val_pairs = train_pairs[:n_val_videos]
    train_pairs = train_pairs[n_val_videos:]
    
    # Test pairs remain separate
    test_pairs = list(zip(test_files_kinect, test_files_mp))
    
    print(f"Train videos: {len(train_pairs)}")
    print(f"Val videos: {len(val_pairs)}")
    print(f"Test videos: {len(test_pairs)}")
    
    return train_pairs, val_pairs, test_pairs

In [35]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import sys
import numpy as np
import os
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import mlflow
import mlflow.pytorch
from sklearn.model_selection import train_test_split

sys.path.append("../../MainProject/Assignment9")
from assignment9_functions import *

random_seed = 112

# Set device
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

# ============================================
# LOAD ALIGNED DATA (YOUR EXISTING CODE)
# ============================================

def load_aligned_pair(mp_file, kinect_file, mp_dir, kinect_dir):
    df_mp = pd.read_csv(os.path.join(mp_dir, mp_file))
    df_kinect = pd.read_csv(os.path.join(kinect_dir, kinect_file))

    df_kinect.columns = df_kinect.columns.str.strip()

    # Align frames
    frames = df_kinect["FrameNo"].values
    df_mp = df_mp[df_mp["FrameNo"].isin(frames)]

    df_mp = df_mp.reset_index(drop=True)
    df_kinect = df_kinect.reset_index(drop=True)

    return df_mp, df_kinect

def input_target_split(dataframe):
    input_cols = [c for c in dataframe.columns if c.endswith('_x') or c.endswith('_y')]
    target_cols = [c for c in dataframe.columns if c.endswith('_z')]
    
    input_data = dataframe[input_cols]
    target_data = dataframe[target_cols]
    return input_data, target_data

kinect_data = "../../MainProject/Assignment9/data/kinect_good_preprocessed"
train_files_kinect, test_files_kinect = split_csvfiles(kinect_data, random_seed, 0.9, 0)

# Load Mediapipe data
mp_data = "../../MainProject/Assignment10/data/csv_of_all_videos"
train_files_mp = [f.replace("kinect", "mediapipe") for f in train_files_kinect]
test_files_mp = [f.replace("kinect", "mediapipe") for f in test_files_kinect]

# Split by VIDEO (no leakage!)
train_pairs, val_pairs, test_pairs = split_by_video(
    train_files_kinect, train_files_mp,
    test_files_kinect, test_files_mp,
    val_ratio=0.15, random_seed=42
)

# Load training data
train_mp_list = []
train_kinect_list = []
for kinect_file, mp_file in train_pairs:
    df_mp, df_kinect = load_aligned_pair(mp_file, kinect_file, mp_data, kinect_data)
    train_mp_list.append(df_mp)
    train_kinect_list.append(df_kinect)

train_data_mp = pd.concat(train_mp_list, ignore_index=True)
train_data_kinect = pd.concat(train_kinect_list, ignore_index=True)

# Load validation data
val_mp_list = []
val_kinect_list = []
for kinect_file, mp_file in val_pairs:
    df_mp, df_kinect = load_aligned_pair(mp_file, kinect_file, mp_data, kinect_data)
    val_mp_list.append(df_mp)
    val_kinect_list.append(df_kinect)

val_data_mp = pd.concat(val_mp_list, ignore_index=True)
val_data_kinect = pd.concat(val_kinect_list, ignore_index=True)

# Load test data (your existing test split)
test_mp_list = []
test_kinect_list = []
for kinect_file, mp_file in test_pairs:
    df_mp, df_kinect = load_aligned_pair(mp_file, kinect_file, mp_data, kinect_data)
    test_mp_list.append(df_mp)
    test_kinect_list.append(df_kinect)

test_data_mp = pd.concat(test_mp_list, ignore_index=True)
test_data_kinect = pd.concat(test_kinect_list, ignore_index=True)

# Now split into X and y
x_train_kinect, y_train_kinect = input_target_split(train_data_kinect)
x_val_kinect, y_val_kinect = input_target_split(val_data_kinect)
x_test_kinect, y_test_kinect = input_target_split(test_data_kinect)

x_train_mp, y_train_mp = input_target_split(train_data_mp)
x_val_mp, y_val_mp = input_target_split(val_data_mp)
x_test_mp, y_test_mp = input_target_split(test_data_mp)

print(f"Train: {len(x_train_mp)} frames")
print(f"Val: {len(x_val_mp)} frames")
print(f"Test: {len(x_test_mp)} frames")

print("="*60)
print("DATA LOADING SUMMARY")
print("="*60)
print(f"MP train rows: {len(x_train_mp)}")
print(f"Kinect train rows: {len(y_train_kinect)}")
print(f"MP test rows: {len(x_test_mp)}")
print(f"Kinect test rows: {len(y_test_kinect)}")

# Convert to tensors
X_train_kinect = torch.tensor(x_train_kinect.values, dtype=torch.float32)
Y_train_kinect = torch.tensor(y_train_kinect.values, dtype=torch.float32)
X_test_kinect = torch.tensor(x_test_kinect.values, dtype=torch.float32)
Y_test_kinect = torch.tensor(y_test_kinect.values, dtype=torch.float32)

X_train_mp = torch.tensor(x_train_mp.values, dtype=torch.float32)
Y_train_mp = torch.tensor(y_train_mp.values, dtype=torch.float32)
X_test_mp = torch.tensor(x_test_mp.values, dtype=torch.float32)
Y_test_mp = torch.tensor(y_test_mp.values, dtype=torch.float32)

# Split train into train/val (80/20)
from sklearn.model_selection import train_test_split

# For Kinect
kinect_train_x, kinect_val_x, kinect_train_y, kinect_val_y = train_test_split(
    X_train_kinect.numpy(), Y_train_kinect.numpy(), test_size=0.2, random_state=random_seed
)
kinect_train_x = torch.tensor(kinect_train_x, dtype=torch.float32)
kinect_val_x = torch.tensor(kinect_val_x, dtype=torch.float32)
kinect_train_y = torch.tensor(kinect_train_y, dtype=torch.float32)
kinect_val_y = torch.tensor(kinect_val_y, dtype=torch.float32)

# For MediaPipe
mp_train_x, mp_val_x, mp_train_y, mp_val_y = train_test_split(
    X_train_mp.numpy(), Y_train_mp.numpy(), test_size=0.2, random_state=random_seed
)
mp_train_x = torch.tensor(mp_train_x, dtype=torch.float32)
mp_val_x = torch.tensor(mp_val_x, dtype=torch.float32)
mp_train_y = torch.tensor(mp_train_y, dtype=torch.float32)
mp_val_y = torch.tensor(mp_val_y, dtype=torch.float32)

print(f"\nKinect - Train: {len(kinect_train_x)}, Val: {len(kinect_val_x)}, Test: {len(X_test_kinect)}")
print(f"MediaPipe - Train: {len(mp_train_x)}, Val: {len(mp_val_x)}, Test: {len(X_test_mp)}")


# ============================================
# DENSE NETWORK ARCHITECTURE
# ============================================

class DenseZPredictor(nn.Module):
    def __init__(self, hidden_layers=[256, 128, 64], dropout=0.3, input_size=26, output_size=13):
        super(DenseZPredictor, self).__init__()
        
        layers = []
        prev_size = input_size
        
        for hidden_size in hidden_layers:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev_size = hidden_size
        
        layers.append(nn.Linear(prev_size, output_size))
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)


# ============================================
# TRAINING FUNCTION WITH MLFLOW
# ============================================

def train_dense_model(model, X_train, Y_train, X_val, Y_val, config, run_name, verbose=True):
    with mlflow.start_run(run_name=run_name) as run:
        mlflow.log_params(config)
        
        train_dataset = TensorDataset(X_train, Y_train)
        val_dataset = TensorDataset(X_val, Y_val)
        
        train_loader = DataLoader(train_dataset, batch_size=config["batch_size"], shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=config["batch_size"], shuffle=False)
        
        criterion = nn.MSELoss()
        optimizer = optim.Adam(model.parameters(), lr=config["learning_rate"], 
                              weight_decay=config.get("weight_decay", 0))
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=20)
        
        best_val_loss = float('inf')
        best_state = None
        patience_counter = 0
        previous_lr = config["learning_rate"]
        history = {'train_loss': [], 'val_loss': [], 'train_mae': [], 'val_mae': []}
        
        for epoch in range(config["epochs"]):
            model.train()
            train_losses, train_maes = [], []
            for batch_X, batch_Y in train_loader:
                batch_X, batch_Y = batch_X.to(device), batch_Y.to(device)
                optimizer.zero_grad()
                predictions = model(batch_X)
                loss = criterion(predictions, batch_Y)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                train_losses.append(loss.item())
                train_maes.append(torch.mean(torch.abs(predictions - batch_Y)).item())
            
            model.eval()
            val_losses, val_maes = [], []
            with torch.no_grad():
                for batch_X, batch_Y in val_loader:
                    batch_X, batch_Y = batch_X.to(device), batch_Y.to(device)
                    predictions = model(batch_X)
                    loss = criterion(predictions, batch_Y)
                    val_losses.append(loss.item())
                    val_maes.append(torch.mean(torch.abs(predictions - batch_Y)).item())
            
            avg_train_loss = np.mean(train_losses)
            avg_train_mae = np.mean(train_maes)
            avg_val_loss = np.mean(val_losses)
            avg_val_mae = np.mean(val_maes)
            
            history['train_loss'].append(avg_train_loss)
            history['val_loss'].append(avg_val_loss)
            history['train_mae'].append(avg_train_mae)
            history['val_mae'].append(avg_val_mae)
            
            scheduler.step(avg_val_loss)
            
            current_lr = optimizer.param_groups[0]['lr']
            if current_lr != previous_lr and verbose:
                print(f"  → Learning rate reduced to {current_lr:.6f}")
            previous_lr = current_lr
            
            if epoch % 20 == 0:
                mlflow.log_metrics({
                    "train_loss": avg_train_loss, "train_mae": avg_train_mae,
                    "val_loss": avg_val_loss, "val_mae": avg_val_mae,
                    "learning_rate": current_lr
                }, step=epoch)
            
            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                best_state = model.state_dict().copy()
                patience_counter = 0
            else:
                patience_counter += 1
            
            if verbose and epoch % 50 == 0:
                print(f"Epoch {epoch:3d}: Train Loss={avg_train_loss:.6f}, Val Loss={avg_val_loss:.6f}, "
                      f"Train MAE={avg_train_mae:.4f}, Val MAE={avg_val_mae:.4f}")
            
            if config.get("patience") and patience_counter >= config["patience"]:
                print(f"Early stopping at epoch {epoch}")
                break
        
        model.load_state_dict(best_state)
        mlflow.pytorch.log_model(model, artifact_path=f"{run_name}_model")
        
    return model, history


def evaluate_model(model, X_test, Y_test, dataset_name=""):
    model.eval()
    with torch.no_grad():
        X_test = X_test.to(device)
        Y_test = Y_test.to(device)
        predictions = model(X_test)
        
        predictions = predictions.cpu().numpy()
        Y_test = Y_test.cpu().numpy()
        
        mse = mean_squared_error(Y_test.flatten(), predictions.flatten())
        mae = mean_absolute_error(Y_test.flatten(), predictions.flatten())
        r2 = r2_score(Y_test.flatten(), predictions.flatten())
        bias = np.mean(predictions.flatten() - Y_test.flatten())
        
        print(f"\n{dataset_name} Test Results:")
        print(f"  MSE: {mse:.6f}")
        print(f"  MAE: {mae:.4f} m ({mae*100:.2f} cm)")
        print(f"  R²:  {r2:.4f}")
        print(f"  Bias: {bias:.6f}")
        
    return {'mse': mse, 'mae': mae, 'r2': r2, 'bias': bias}


# ============================================
# CONFIGURATION
# ============================================

config = {
    "hidden_layers": [256, 128, 64],
    "dropout": 0.3,
    "learning_rate": 0.001,
    "batch_size": 64,
    "epochs": 300,
    "weight_decay": 1e-5,
    "patience": 30,
}


# ============================================
# TRAIN ON KINECT DATA
# ============================================

print("\n" + "="*60)
print("TRAINING ON KINECT DATA (Kinect XY → Kinect Z)")
print("="*60)

kinect_model = DenseZPredictor(
    hidden_layers=config["hidden_layers"],
    dropout=config["dropout"]
).to(device)

print(f"Parameters: {sum(p.numel() for p in kinect_model.parameters()):,}")

kinect_model, kinect_history = train_dense_model(
    kinect_model, 
    kinect_train_x.to(device), kinect_train_y.to(device),
    kinect_val_x.to(device), kinect_val_y.to(device),
    config, "Kinect_XY_to_Z", verbose=True
)

kinect_results = evaluate_model(kinect_model, X_test_kinect, Y_test_kinect, "Kinect")


# ============================================
# TRAIN ON MEDIAPIPE DATA
# ============================================

print("\n" + "="*60)
print("TRAINING ON MEDIAPIPE DATA (MediaPipe XY → MediaPipe Z)")
print("="*60)

mp_model = DenseZPredictor(
    hidden_layers=config["hidden_layers"],
    dropout=config["dropout"]
).to(device)

print(f"Parameters: {sum(p.numel() for p in mp_model.parameters()):,}")

mp_model, mp_history = train_dense_model(
    mp_model,
    mp_train_x.to(device), mp_train_y.to(device),
    mp_val_x.to(device), mp_val_y.to(device),
    config, "MediaPipe_XY_to_Z", verbose=True
)

mp_results = evaluate_model(mp_model, X_test_mp, Y_test_mp, "MediaPipe")


# ============================================
# COMPARE RESULTS
# ============================================

print("\n" + "="*60)
print("COMPARISON: Kinect vs MediaPipe")
print("="*60)

comparison_df = pd.DataFrame({
    "Metric": ["MSE", "MAE (m)", "MAE (cm)", "R²", "Bias"],
    "Kinect": [f"{kinect_results['mse']:.6f}", f"{kinect_results['mae']:.4f}", 
               f"{kinect_results['mae']*100:.2f}", f"{kinect_results['r2']:.4f}", 
               f"{kinect_results['bias']:.6f}"],
    "MediaPipe": [f"{mp_results['mse']:.6f}", f"{mp_results['mae']:.4f}", 
                  f"{mp_results['mae']*100:.2f}", f"{mp_results['r2']:.4f}", 
                  f"{mp_results['bias']:.6f}"]
})
print(comparison_df.to_string(index=False))

# Plot training curves
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes[0,0].plot(kinect_history['train_loss'], label='Train')
axes[0,0].plot(kinect_history['val_loss'], label='Val')
axes[0,0].set_title('Kinect - Loss')
axes[0,0].legend(); axes[0,0].grid(True)
axes[0,1].plot(kinect_history['train_mae'], label='Train')
axes[0,1].plot(kinect_history['val_mae'], label='Val')
axes[0,1].set_title('Kinect - MAE')
axes[0,1].legend(); axes[0,1].grid(True)
axes[1,0].plot(mp_history['train_loss'], label='Train')
axes[1,0].plot(mp_history['val_loss'], label='Val')
axes[1,0].set_title('MediaPipe - Loss')
axes[1,0].legend(); axes[1,0].grid(True)
axes[1,1].plot(mp_history['train_mae'], label='Train')
axes[1,1].plot(mp_history['val_mae'], label='Val')
axes[1,1].set_title('MediaPipe - MAE')
axes[1,1].legend(); axes[1,1].grid(True)
plt.tight_layout()
plt.savefig('dense_comparison.png', dpi=150)
plt.show()

print("\nModels saved as 'kinect_dense_model.pt' and 'mediapipe_dense_model.pt'")
torch.save(kinect_model.state_dict(), "kinect_dense_model.pt")
torch.save(mp_model.state_dict(), "mediapipe_dense_model.pt")

Using device: mps
Train videos: 137
Val videos: 24
Test videos: 18
Train: 18151 frames
Val: 3542 frames
Test: 2312 frames
DATA LOADING SUMMARY
MP train rows: 18151
Kinect train rows: 18151
MP test rows: 2312
Kinect test rows: 2312

Kinect - Train: 14520, Val: 3631, Test: 2312
MediaPipe - Train: 14520, Val: 3631, Test: 2312

TRAINING ON KINECT DATA (Kinect XY → Kinect Z)
Parameters: 48,909
Epoch   0: Train Loss=0.008838, Val Loss=0.004785, Train MAE=0.0686, Val MAE=0.0508


KeyboardInterrupt: 

In [36]:
# === CROSS VALIDATION + MLFLOW INTEGRATION ===

import numpy as np
from sklearn.model_selection import KFold
import torch
import torch.optim as optim
import mlflow
import mlflow.pytorch

# ---- MLFLOW SETUP ----
mlflow.set_experiment("video-cross-validation")

# 1. KFold split på videonivå

def cross_validate_videos(train_files_kinect, train_files_mp, k=5, random_seed=42):
    pairs = list(zip(train_files_kinect, train_files_mp))
    pairs = np.array(pairs)

    kf = KFold(n_splits=k, shuffle=True, random_state=random_seed)

    folds = []

    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(pairs)):
        train_pairs = pairs[train_idx]
        val_pairs = pairs[val_idx]

        train_kinect, train_mp = zip(*train_pairs)
        val_kinect, val_mp = zip(*val_pairs)

        folds.append({
            "train_kinect": list(train_kinect),
            "train_mp": list(train_mp),
            "val_kinect": list(val_kinect),
            "val_mp": list(val_mp)
        })

    return folds


# 2. Kör cross validation med MLFLOW

def run_cross_validation(train_files_kinect, train_files_mp, k=5, device="cuda"):
    folds = cross_validate_videos(train_files_kinect, train_files_mp, k=k)

    all_results = []

    with mlflow.start_run(run_name="cross_validation"):
        mlflow.log_param("num_folds", k)

        for i, fold in enumerate(folds):
            print(f"\n===== Fold {i+1}/{k} =====")

            train_kinect = fold["train_kinect"]
            train_mp = fold["train_mp"]
            val_kinect = fold["val_kinect"]
            val_mp = fold["val_mp"]

            with mlflow.start_run(run_name=f"fold_{i+1}", nested=True):
                mlflow.log_param("fold", i+1)

                # DataLoaders
                train_loader = create_dataloader(train_kinect, train_mp)
                val_loader = create_dataloader(val_kinect, val_mp)

                # Modell
                model = MyModel().to(device)
                optimizer = optim.Adam(model.parameters(), lr=1e-3)

                mlflow.log_param("learning_rate", 1e-3)

                # Träna
                train_model(model, train_loader, val_loader, optimizer)

                # Utvärdera
                metrics = evaluate_model(model, val_loader)
                print(f"Fold {i+1} metrics:", metrics)

                # Logga metrics
                for key, value in metrics.items():
                    mlflow.log_metric(key, float(value))

                # Logga modell
                mlflow.pytorch.log_model(model, f"model_fold_{i+1}")

                all_results.append(metrics)

        # Logga summary
        summary = summarize_results(all_results)
        for key, stats in summary.items():
            mlflow.log_metric(f"{key}_mean", stats["mean"])
            mlflow.log_metric(f"{key}_std", stats["std"])

    return all_results


# 3. Summera resultat

def summarize_results(all_results):
    keys = all_results[0].keys()

    summary = {}

    for key in keys:
        values = [r[key] for r in all_results]
        summary[key] = {
            "mean": float(np.mean(values)),
            "std": float(np.std(values))
        }

    print("\n===== CROSS VALIDATION RESULT =====")
    for k, v in summary.items():
        print(f"{k}: mean={v['mean']:.4f}, std={v['std']:.4f}")

    return summary


# === DIREKT KÖRBAR EXEMPEL ===

# train_files_kinect = [...]
# train_files_mp = [...]

# results = run_cross_validation(train_files_kinect, train_files_mp, k=5, device="cuda")


In [37]:
results = run_cross_validation(train_files_kinect, train_files_mp, k=5, device="cuda")


===== Fold 1/5 =====


NameError: name 'create_dataloader' is not defined